In [1]:
import sys
from pathlib import Path
_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(_root / "scripts"))

import numpy as np, pandas as pd, geopandas as gpd
import folium
from folium.plugins import MiniMap, Fullscreen
from glacier_utils import ROOT, PROC, AEA, SIZE_LABELS, SIZE_COLORS, add_size_class

SECTORS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

gdf = add_size_class(gpd.read_file(PROC / "nepal_glaciers.gpkg", layer="glaciers"))
core = gdf[gdf["centroid_in_nepal"]].copy()
tb = gdf[gdf["transboundary"]].copy()
print(f"Nepal glaciers: {len(core):,} | transboundary: {len(tb):,}")

Nepal glaciers: 4,480 | transboundary: 199


In [2]:
SIMPLIFY_M = 30      # simplification tolerance in metres; raise to 50-100 if the HTML is too big

def prep(df):
    """Simplify in the equal-area CRS, round coordinates, keep only the fields the map needs."""
    d = df.to_crs(AEA)
    d["geometry"] = d.geometry.simplify(SIMPLIFY_M, preserve_topology=True)
    d = d.to_crs("EPSG:4326")
    d["geometry"] = d.geometry.set_precision(1e-5)          # about 1 m, trims decimals in the GeoJSON
    d = d[~d.geometry.is_empty].copy()

    d["aspect"] = [f"{SECTORS[int(((a + 22.5) % 360) // 45)]} ({a:.0f}°)" for a in d["aspect_deg"]]
    d["gname"] = d["glac_name"].fillna("(unnamed)")
    d["area"] = d["area_km2"].round(3)
    for c in ("zmin_m", "zmed_m", "zmax_m"):
        d[c] = d[c].round(0).astype(int)
    d["src_year"] = d["src_year"].astype(int)
    d["size_class"] = d["size_class"].astype(str)
    keep = ["rgi_id", "gname", "size_class", "area", "zmin_m", "zmed_m", "zmax_m", "aspect", "src_year"]
    return d[keep + ["geometry"]].reset_index(drop=True)

core_s, tb_s = prep(core), prep(tb)

mb = (len(core_s.to_json()) + len(tb_s.to_json())) / 1e6
print(f"Features kept: {len(core_s):,} + {len(tb_s):,} | GeoJSON payload at {SIMPLIFY_M} m tolerance: {mb:.1f} MB")

Features kept: 4,480 + 199 | GeoJSON payload at 30 m tolerance: 4.4 MB


In [5]:
# ---- Fresh tooltip/popup objects per layer (sharing one object across layers breaks Folium) ----
def make_tooltip():
    return folium.GeoJsonTooltip(
        fields=["rgi_id", "area", "zmin_m", "zmax_m", "aspect"],
        aliases=["RGI ID", "Area (km²)", "Min elevation (m)", "Max elevation (m)", "Aspect"],
        sticky=False, style="font-size:12px;")

def make_popup():
    return folium.GeoJsonPopup(
        fields=["rgi_id", "gname", "size_class", "area", "zmin_m", "zmed_m", "zmax_m", "aspect", "src_year"],
        aliases=["RGI ID", "Name", "Size class (km²)", "Area (km²)", "Min elevation (m)",
                 "Median elevation (m)", "Max elevation (m)", "Aspect", "Outline year"],
        max_width=280)

# ---- Sanity checks before building ----
print("Features per size class in the simplified data:")
print(core_s["size_class"].value_counts().reindex(SIZE_LABELS).to_string())
print("Geometry types:", core_s.geometry.geom_type.value_counts().to_dict())
print("Bounds:", core_s.total_bounds.round(2))
assert len(core_s) == len(core) and len(tb_s) == len(tb), "glaciers lost during simplification"

# ---- Base map and basemaps ----
m = folium.Map(location=[28.3, 84.0], zoom_start=7, tiles=None, control_scale=True, prefer_canvas=True)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/Canvas/World_Light_Gray_Base/MapServer/tile/{z}/{y}/{x}",
    name="Light gray (Esri)", max_zoom=16,
    attr="Tiles © Esri, HERE, Garmin, FAO, NOAA, USGS, OpenStreetMap contributors").add_to(m)
folium.TileLayer("OpenStreetMap", name="OpenStreetMap", show=False).add_to(m)
folium.TileLayer(
    tiles="https://tile.opentopomap.org/{z}/{x}/{y}.png", name="Topographic (OpenTopoMap)", show=False, max_zoom=17,
    attr="Map data © OpenStreetMap contributors, SRTM | Style © OpenTopoMap (CC-BY-SA)").add_to(m)
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    name="Satellite (Esri)", show=False, max_zoom=18,
    attr="Tiles © Esri, Maxar, Earthstar Geographics and the GIS User Community").add_to(m)

# ---- Nepal boundary ----
bnd = gpd.read_file(ROOT / "data/raw/boundary/geoBoundaries-NPL-ADM0-all/geoBoundaries-NPL-ADM0_simplified.shp")[["geometry"]]
folium.GeoJson(bnd, name="Nepal boundary",
               style_function=lambda f: {"fill": False, "color": "#222222", "weight": 1.6}).add_to(m)

# ---- One layer per size class ----
stats = core.groupby("size_class", observed=False).agg(n=("rgi_id", "size"), area=("area_km2", "sum"))
for label, color in zip(SIZE_LABELS, SIZE_COLORS):
    sub = core_s[core_s["size_class"] == label]
    fg = folium.FeatureGroup(name=f"{label} km² ({len(sub):,} glaciers)", show=True)
    folium.GeoJson(
        sub,
        style_function=lambda f, c=color: {"fillColor": c, "color": "#333333", "weight": 0.7, "fillOpacity": 0.85},
        highlight_function=lambda f: {"weight": 2.5, "color": "#D55E00", "fillOpacity": 1},
        tooltip=make_tooltip(), popup=make_popup()).add_to(fg)
    fg.add_to(m)

# ---- Transboundary glaciers ----
fg_tb = folium.FeatureGroup(name=f"Transboundary, centre outside Nepal ({len(tb_s):,})", show=True)
folium.GeoJson(
    tb_s,
    style_function=lambda f: {"fillColor": "#bdbdbd", "color": "#555555", "weight": 0.8,
                              "fillOpacity": 0.6, "dashArray": "3"},
    highlight_function=lambda f: {"weight": 2.5, "color": "#D55E00"},
    tooltip=make_tooltip(), popup=make_popup()).add_to(fg_tb)
fg_tb.add_to(m)

# ---- Info panel ----
hyp = pd.read_parquet(PROC / "nepal_hypsometry_km2.parquet").set_index("rgi_id")
hyp.columns = hyp.columns.astype(int)
A50 = hyp.loc[core["rgi_id"]].sum()
cum = A50.cumsum().to_numpy() / A50.sum()
i = int(np.searchsorted(cum, 0.5))
prev = cum[i - 1] if i > 0 else 0
med_elev = (A50.index[i] - 25) + 50 * (0.5 - prev) / (cum[i] - prev)

pct_small = 100 * (core["area_km2"] < 1).mean()
y0, y1 = int(core["src_year"].min()), int(core["src_year"].max())
rows = "".join(
    f'<div><span style="display:inline-block;width:14px;height:10px;background:{c};border:1px solid #333;margin-right:6px"></span>'
    f'{lab} km² — {int(stats.loc[lab, "n"]):,} glaciers, {stats.loc[lab, "area"]:,.0f} km²</div>'
    for lab, c in zip(SIZE_LABELS, SIZE_COLORS))

panel = f"""
<div style="position:fixed; bottom:20px; left:10px; z-index:9999; background:rgba(255,255,255,0.93);
            padding:10px 12px; border-radius:6px; box-shadow:0 1px 5px rgba(0,0,0,.4);
            font:12px/1.45 Arial,sans-serif; max-width:290px;">
  <div style="font-size:14px; font-weight:bold; margin-bottom:4px;">Nepal glacier inventory (RGI 7.0)</div>
  <div>{len(core):,} glaciers, {core['area_km2'].sum():,.0f} km² · {pct_small:.0f}% are smaller than 1 km²<br>
       Area-weighted median elevation: {med_elev:,.0f} m</div>
  <div style="margin:6px 0 2px; font-weight:bold;">Glacier size class</div>
  {rows}
  <div><span style="display:inline-block;width:14px;height:10px;background:#bdbdbd;border:1px dashed #555;margin-right:6px"></span>
       Transboundary ({len(tb):,}; centre point outside Nepal)</div>
  <div style="margin-top:6px; font-size:10.5px; color:#444;">
    Outlines: RGI 7.0 Consortium (2023), doi:10.5067/f6jmovy5navz; outline dates {y0}–{y1}.
    Boundary: geoBoundaries. Outlines simplified ({SIMPLIFY_M} m) for display.</div>
</div>"""
m.get_root().html.add_child(folium.Element(panel))
m.get_root().header.add_child(folium.Element("<title>Nepal Glacier Explorer</title>"))

# ---- Controls and view ----
MiniMap(toggle_display=True, position="bottomright").add_to(m)
Fullscreen().add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
b = core_s.total_bounds
m.fit_bounds([[b[1], b[0]], [b[3], b[2]]])

out = ROOT / "docs" / "index.html"
m.save(out)
print(f"\nSaved {out.relative_to(ROOT)} ({out.stat().st_size / 1e6:.1f} MB)")

Features per size class in the simplified data:
size_class
<0.5     3092
0.5–1     608
1–5       640
5–10       69
10–50      67
>50         4
Geometry types: {'Polygon': 4480}
Bounds: [80.53 27.53 88.2  30.47]

Saved docs\index.html (4.9 MB)
